# Workflows

A **workflow** is a high-level wrapper around an approximator that manages the full inference pipeline: data generation, compilation, training, checkpointing, sampling, and diagnostics. For most use cases a workflow is the most convenient entry point — it handles the boilerplate so you can focus on your model.

BayesFlow provides three workflow classes:

| Workflow | Use case |
|---|---|
| `BasicWorkflow` | Standard posterior / likelihood estimation (NPE, NLE, scoring rules) |
| `EnsembleWorkflow` | Train and query multiple approximators jointly |
| `CompositionalWorkflow` | Compositional / hierarchical inference with a `DiffusionModel` |

All three live in `bayesflow.workflows` and are re-exported from the top-level `bayesflow` namespace. This page covers each in turn, then explains the training strategies (`fit_online`, `fit_offline`, `fit_disk`) and the diagnostic utilities common to all workflows.

## BasicWorkflow

`BasicWorkflow` is the standard starting point. At a minimum you provide a simulator and tell it which variables play which role; the workflow builds a default adapter, compiles the approximator, and is ready to train.

```python
import bayesflow as bf

workflow = bf.BasicWorkflow(
    simulator=simulator,
    inference_variables=["theta"],   # what you want to infer
    summary_variables=["x"],         # data to compress through the summary network
    inference_network=bf.networks.FlowMatching(),
    summary_network=bf.networks.TimeSeriesTransformer(summary_dim=16),
)

history = workflow.fit_online(epochs=100, batch_size=128, num_batches_per_epoch=500)
```

### What `BasicWorkflow` builds for you

- A default `Adapter` that routes `inference_variables`, `summary_variables`, and `inference_conditions` correctly.
- An approximator (`ContinuousApproximator` or `ScoringRuleApproximator` depending on the inference network).
- An `Adam` optimizer with a cosine decay schedule appropriate for the chosen training strategy.

You can override any of these by passing a custom `adapter`, `optimizer`, or a pre-built approximator via `inference_network`.

### Key constructor arguments

| Argument | Default | Description |
|---|---|---|
| `simulator` | `None` | Source of training data |
| `inference_variables` | — | Variables to infer (e.g., `["theta"]`) |
| `summary_variables` | `None` | Variables to summarize with the summary network |
| `inference_conditions` | `None` | Variables fed directly to the inference network (not summarized) |
| `inference_network` | `"coupling_flow"` | Network or name string |
| `summary_network` | `None` | Summary network or name string |
| `initial_learning_rate` | `5e-4` | Starting LR for the optimizer |
| `checkpoint_filepath` | `None` | Directory for automatic checkpointing |
| `checkpoint_name` | `"model"` | File stem; saved as `{name}.keras` |
| `save_best_only` | `False` | Keep only the lowest-loss checkpoint |

### After training

The trained approximator is always accessible at `workflow.approximator`. All approximator methods (`.sample()`, `.log_prob()`, etc.) work directly on it:

```python
# High-level workflow interface
samples = workflow.sample(num_samples=1000, conditions={"x": x_obs})

# Equivalent — directly on the approximator
samples = workflow.approximator.sample(num_samples=1000, conditions={"x": x_obs})
```

To save the model, use the approximator (not the workflow):

```python
workflow.approximator.save("my_model.keras")
```

## EnsembleWorkflow

`EnsembleWorkflow` trains multiple approximators jointly on the same data, then lets you query them individually or as a merged mixture. This is useful for uncertainty quantification over the inference network itself, or for comparing different network architectures on the same problem.

### Construction modes

**Size mode** — clone one network $N$ times automatically:

```python
workflow = bf.EnsembleWorkflow(
    simulator=simulator,
    inference_variables=["theta"],
    summary_variables=["x"],
    inference_networks=bf.networks.FlowMatching(),  # will be cloned ensemble_size times
    ensemble_size=5,
)
```

**Dictionary mode** — give each member an explicit name and network:

```python
workflow = bf.EnsembleWorkflow(
    simulator=simulator,
    inference_variables=["theta"],
    summary_variables=["x"],
    inference_networks={
        "flow": bf.networks.FlowMatching(),
        "coupling": bf.networks.CouplingFlow()
    },
    summary_networks={
        "flow": bf.networks.SetTransformer(summary_dim=16),
        "coupling": bf.networks.SetTransformer(summary_dim=16)
    },
)
```

### Training

`EnsembleWorkflow` exposes the same `fit_online` / `fit_offline` / `fit_disk` interface as `BasicWorkflow`, plus a `data_reuse` parameter that controls data sharing between members:

```python
# data_reuse=1.0 (default): all members see identical batches
# data_reuse=0.0: each member draws a fully independent batch
history = workflow.fit_online(
    epochs=100,
    batch_size=128,
    num_batches_per_epoch=500,
    data_reuse=0.8,  # 80% shared data across members
)
```

### Inference

```python
# Merged mixture samples (default)
samples = workflow.sample(num_samples=500, conditions={"x": x_obs})

# Per-member samples
samples = workflow.sample(
    num_samples=500,
    conditions={"x": x_obs},
    merge_members=False,   # returns {"flow": ..., "coupling": ...}
)

# Weighted mixture
samples = workflow.sample(
    num_samples=500,
    conditions={"x": x_obs},
    member_weights={"flow": 0.6, "coupling": 0.4},
)

# Log-probability under the mixture
log_p = workflow.log_prob(data={"x": x_obs, "theta": theta})
```

## CompositionalWorkflow

`CompositionalWorkflow` extends `BasicWorkflow` for **compositional inference** — settings where you want to draw posterior samples conditioned on multiple datasets simultaneously (e.g., combining evidence from several experiments or performing joint inference over a hierarchical structure). The inference network must be a `DiffusionModel`.

### Basic usage

```python
workflow = bf.CompositionalWorkflow(
    simulator=simulator,
    inference_variables=["theta"],
    inference_conditions=["x"],
    inference_network=bf.networks.DiffusionModel()  # required
)

history = workflow.fit_online(epochs=200, batch_size=64, num_batches_per_epoch=500)
```

### Compositional sampling

Conditions are expected to have shape `(n_datasets, n_compositional, ...)` — the second axis holds the multiple datasets being composed.

```python
# x_obs has shape (n_datasets, n_compositional, obs_dim)
samples = workflow.compositional_sample(
    num_samples=500,
    conditions={"x": x_obs},
)
```

You can also inject a prior score function to guide compositional sampling:

```python
def prior_score(data, t=None):
    """Score of a Gaussian prior on theta."""
    return {"theta": -data["theta"] / prior_variance}

samples = workflow.compositional_sample(
    num_samples=500,
    conditions={"x": x_obs},
    compute_prior_score=prior_score,
)
```

### Building from a trained BasicWorkflow

If you already have a trained `BasicWorkflow` with a `DiffusionModel`, you can promote it to a `CompositionalWorkflow` without retraining. The network weights are cloned and transferred:

```python
# basic_workflow was already trained
compositional_workflow = bf.CompositionalWorkflow.from_basic_workflow(
    basic_workflow,
    simulator=simulator,  # override any constructor arguments as needed
)

samples = compositional_workflow.compositional_sample(
    num_samples=500,
    conditions={"x": x_obs},
)
```

See the [Compositional Diffusion example](https://github.com/bayesflow-org/bayesflow/blob/dev/examples/Compositional_Diffusion.ipynb) for a worked end-to-end demonstration.

## Training Strategies

All three workflow classes support the same three training strategies. Choose based on how you generate data:

### `fit_online` — simulate during training

The simulator is called at each training step. This requires no pre-generated dataset and prevents overfitting to a fixed sample pool. Use this when simulation is fast (seconds per batch).

```python
history = workflow.fit_online(
    epochs=100,
    num_batches_per_epoch=500,  # simulator calls per epoch
    batch_size=128,
    validation_data=1000,       # generate N validation sets from the simulator
)
```

### `fit_offline` — train on a pre-generated dataset in memory

Simulate everything upfront and store it in a dictionary. Training is faster (no simulator overhead per step), but the model trains on a fixed finite dataset.

```python
# Pre-generate data
data = simulator.sample((50_000,))   # dict of arrays, 50k simulations

history = workflow.fit_offline(
    data=data,
    epochs=200,
    batch_size=128,
    validation_data={"theta": ..., "x": ...},  # or an integer
)
```

### `fit_disk` — stream from files on disk

For very large simulation budgets that don't fit in memory. Data is stored as `.pkl` files (or any format with a custom `load_fn`) and loaded lazily during training.

```python
# Each file in /data/simulations/ is a dict of arrays for one batch
history = workflow.fit_disk(
    root="/data/simulations",
    pattern="*.pkl",    # glob pattern to find files
    batch_size=128,
    epochs=100
)
```

### Augmentations

All fit methods accept an `augmentations` argument — functions applied to each batch *before* the adapter, useful for data augmentation during training only (e.g., adding noise, random scaling):

```python
def add_noise(batch):
    batch["x"] = batch["x"] + np.random.normal(0, 0.01, batch["x"].shape)
    return batch

workflow.fit_online(epochs=100, augmentations=add_noise)
```

## Diagnostics

All workflows expose built-in diagnostic utilities that run automatically against test data.

### Default diagnostics

```python
# Recommended: obtain samples from test data
samples = workflow.sample(conditions=test_data, num_samples=1000, batch_size=100)


figures = workflow.plot_default_diagnostics(samples)
# figures is a dict: {"losses", "recovery", "calibration_ecdf", "coverage", "z_score_contraction"}
```

Individual plots:
- **`losses`** — training (and validation) loss over epochs
- **`recovery`** — posterior mean vs. ground-truth parameter ("how well does the posterior center on the truth?")
- **`calibration_ecdf`** — empirical coverage: marginal posteriors should be uniformly calibrated
- **`coverage`** — joint credible-interval coverage across parameter dimensions
- **`z_score_contraction`** — how much the posterior contracts relative to the prior

> **`inference_variables` is required for automated diagnostics.** Make sure to pass `inference_variables=["theta"]` (or your parameter names) when constructing the workflow; otherwise the diagnostic methods cannot automatically identify the targets.

### Accessing metrics numerically

```python
metrics_df = workflow.compute_default_diagnostics(samples, as_data_frame=True)
```

### Custom diagnostics

Pass a dictionary of `{name: plot_fn}` for custom visualizations:

```python
def my_plot(estimates, targets, variable_keys=None, variable_names=None, **kwargs):
    # estimates, targets are dicts of arrays
    ...
    return fig  # matplotlib Figure

figures = workflow.plot_custom_diagnostics(samples, plot_fns={"my_plot": my_plot})
```